# Population Response Coding Model: Parameter-Fitting Walkthrough

This notebook gives a guided walkthrough of the main features of the parameter-fitting script: `ParamFits.py` 

The script fits population-code model parameters to behavioural data by repeatedly simulating model output, comparing it with target data, and using differential evolution to minimise the fit error.

## 1. What the script is trying to do

At a high level, the script asks:

> Which model parameters make the simulated behavioural scores best match the empirical behavioural scores?

The main fitting targets are either `OEI` values or threshold values. The code can fit:

1. group-average behavioural data;
2. individual-participant behavioural data;
3. one task at a time, or both tasks jointly;
4. different decoder rules for the two tasks, contrast detection (`sum`, `mag`, `peak`) and orientation discrimination (`WTA`, `PV`, `ML`).

The optimisation itself is handled using `scipy.optimize.differential_evolution`, which searches over bounded parameter ranges for the parameter values associated with the lowest error.

## 2. Main imports and dependencies

The script combines standard scientific Python tools with custom project modules. The custom modules are where most of the model-specific simulation logic lives.

In [11]:
import copy
from copy import deepcopy
import itertools as itls
import os
from multiprocessing import Pool
import numpy as np
from numpy import asarray as arr
import pandas as pd
from scipy.optimize import differential_evolution
import time

# Custom analysis utilities
from mytools.analysis import get_ave_data
from mytools.dframe_tools import custom_sort

# Model-specific components
from components.model_simulation import ModelSimulator

## 3. The central optimisation loop

The optimisation pipeline is organised around three main functions/classes:

| Component | Role                                                                    |
|---|-------------------------------------------------------------------------|
| `run_de()` | Runs one differential-evolution search for a given random seed.         |
| `run_objective_fn()` | Evaluates one candidate parameter vector and returns its error.         |
| `ObjectiveFunction` | Runs the model simulation and computes the error from behavioural data. |


In [12]:
def run_de(objH, param_names, bounds, popsize, maxiter, seed=None):
    """Run one differential-evolution search."""

    de_result = differential_evolution(
        lambda x: run_objective_fn(
            x,
            param_names,
            objH,
            n_eval=1, 
            n_level_reps=50,  # Number of repeats for each point along the psychometric function
            append_all_fits=True
        ),
        bounds=bounds,  # Search within bounded parameter range
        
        popsize=popsize,  # Controls search diversity by setting how many candidate solutions are used per parameter.
        maxiter=maxiter,  # Maximum number of generations/iterations before stopping.
        
        polish=False,
        seed=seed
    )

    return de_result

### Why several differential-evolution runs?

The script runs differential evolution multiple times using different random seeds. This reduces dependence on a single stochastic search path. After these runs, all candidate fits are pooled, and the best-performing subset is retained for more careful re-evaluation.

## 4. The objective function

`run_objective_fn()` is the bridge between the optimiser and the model. It receives one proposed parameter vector, runs the model, extracts the resulting error, and returns that error to the optimiser.

A useful detail is that parameters ending in `_fraction` are fitted on a log scale. This makes ratios such as `1/4` and `4` symmetric around zero after transformation.

In [13]:
def run_objective_fn(param_values, param_names, objH, n_eval, n_level_reps=None,
                     append_all_fits=False, _return_output=False):
    """Evaluate one candidate parameter vector."""

    # Change simulation repetitions if requested.
    if n_level_reps is not None:
        objH.config['n_level_reps'] = n_level_reps

    # Run the model and compute the fit error.
    model_output, var_params = objH.run(param_values, param_names, n_eval)
    total_error = model_output['overall_fit_error'].iloc[0]

    # Save full output if this candidate should be retained for later analysis.
    if append_all_fits:
        global all_fits_list
        all_fits_list.append(model_output)

    if _return_output:
        return model_output
    else:
        return total_error

## 5. How model error is computed

The `ObjectiveFunction` class runs simulations for each task/decoder pair. For each pair, it:

1. simulates model performance;
2. extracts the requested measure, such as `OEI` or `thresholds`;
3. retrieves the matching behavioural target values;
4. computes mean squared error;
5. stores the error alongside the model output.

This is the part of the script that decides whether a parameter set is a good or poor fit.

In [14]:
class ObjectiveFunction:
    """Computes error between model output and target data."""

    def run(self, param_values, param_names, n_eval=1):
        var_params = {name: value for name, value in zip(param_names, param_values)}
        all_model_output = []

        for i_task, i_decoder in zip(self.tasknames, self.decoders):
            results = [
                self.simulator.simulate(
                    i_task,
                    i_decoder,
                    self.measure,
                    var_params,
                    self.config,
                    n_eval
                )
            ]

            model_output = pd.concat([pd.DataFrame(r) for r in results])

            if self.measure == 'thresholds':
                model_scores = []
                for c, o in zip(model_output['cardinalT'], model_output['obliqueT']):
                    model_scores = [*model_scores, c, o]
            else:
                model_scores = arr(model_output[self.measure])

            target_scores = np.hstack(self.config[self.measure][i_task])
            task_error = np.mean((model_scores - target_scores) ** 2)

            model_output['overall_fit_error'] = task_error
            all_model_output.append(model_output)

        return pd.concat(all_model_output), var_params

## 6. Full optimisation strategy

The `optimize_parameters()` function manages the full fitting workflow.

The key idea is to use a relatively fast, noisy optimisation stage first, then re-run the best candidate fits with more repetitions to obtain more stable estimates.

The stages are:

1. define free-parameter names and bounds;
2. run differential evolution several times;
3. pool all fits across runs;
4. keep the top 5% of candidate fits;
5. re-run the top fits with more repetitions;
6. remove unstable outliers;
7. take the median parameter values from the retained fits;
8. run and save the final model fit.

## 7. Behavioural targets and simulation settings

The `config` dictionary contains both simulation settings and behavioural target data. This is important because the model output is scored directly against these target values.

For example, when `MEASURE = 'OEI'`, the optimiser compares simulated OEI values against the entries in `config['OEI']`.

In [15]:
MEASURE = 'OEI'
CONTRAST = 100
TOTAL_N = 540

config = {
    'run_type': 'loop',
    'design': 'mocs',
    'std_oris': [0, 45],
    'contrasts': [CONTRAST],
    'n_level_reps': 10,
    'step_size': 0.02,

    'OEI': {
        'con_det': [1.06, 1.39, 1.71],
        'ori_dis': [2.45, 3.25, 3.36]
    },

    'thresholds': {
        'con_det': [[0.459, 0.491], [1.123, 1.792], [5.154, 9.499]],
        'ori_dis': [[0.817, 2.033], [1.016, 3.261], [1.339, 4.049]]
    }
}

## 8. Spatial-frequency conditions

The script uses three spatial-frequency conditions: `low`, `mid`, and `high`. Each condition has a different tuning width parameter.

These condition labels allow the output to be grouped and summarised by condition after fitting.

In [16]:
cond_labels = ['low', 'mid', 'high']
sf_params = {'fwhm': [57.6, 30.4, 22.2]}  # Bandwidth here is defined as full-width half max

## 9. Free parameters and model variants

The script does not fit just one model. It generates every non-empty combination of four anisotropy parameters:

- `n_fraction`
- `fwhm_fraction`
- `noise_scale_fraction`
- `rmax_fraction`

Each combination of anisotropies can be treated as a separate model variant. This allows us to compare whether different sources of anisotropy are sufficient to explain the behavioural data.

In [17]:
allFreeParams = [
    'n_fraction',
    'fwhm_fraction',
    'noise_scale_fraction',
    'rmax_fraction'
]

freeParamCombos = []
for r in range(1, len(allFreeParams) + 1):
    freeParamCombos.extend(itls.combinations(allFreeParams, r))

freeParamCombos[:5], len(freeParamCombos)

([('n_fraction',),
  ('fwhm_fraction',),
  ('noise_scale_fraction',),
  ('rmax_fraction',),
  ('n_fraction', 'fwhm_fraction')],
 15)

## 10. Log-transforming fraction parameters

Fraction parameters are ratios, for example cardinal:oblique ratios. The script log-transforms these parameters before fitting.

This has two advantages:

1. ratios below and above 1 become symmetric around zero;
2. the optimiser searches multiplicative changes more naturally.

For example, `1/4` and `4` become equally far from zero on the log scale.

In [18]:
free_params_settings = {
    'n_fraction': {'x0': 1, 'bounds': (1 / 4, 4)},
    'fwhm_fraction': {'x0': 1, 'bounds': (1 / 4, 4)},
    'rmax_fraction': {'x0': 1, 'bounds': (1 / 4, 4)},
    'noise_scale_fraction': {'x0': 1, 'bounds': (1 / 4, 4)}
}

# Example: log-transform bounds and initial value.
example = copy.deepcopy(free_params_settings['n_fraction'])
for key in example:
    example[key] = np.log(example[key])

example

{'x0': np.float64(0.0), 'bounds': array([-1.386,  1.386])}

## 11. Average-data fitting branch

When `FIT_AVERAGE = True`, the script fits the model to the group-average behavioural values in `config`.

If `FIT_BOTH_TASKS = False`, each task and decoder is fitted separately. 

If `FIT_BOTH_TASKS = True`, the script fits both tasks together for each pair of decoders.

In [19]:
FIT_AVERAGE = True
FIT_BOTH_TASKS = False

DECODERS = {
    'con_det': ['sum', 'mag', 'peak'],
    'ori_dis': ['WTA', 'PV', 'ML']
}
TASK_NAMES = ['ori_dis', 'con_det']

# Separate-task fitting pattern.
for task in TASK_NAMES:
    for decoder in DECODERS[task]:
        print(f'Would fit task={task}, decoder={decoder}')

Would fit task=ori_dis, decoder=WTA
Would fit task=ori_dis, decoder=PV
Would fit task=ori_dis, decoder=ML
Would fit task=con_det, decoder=sum
Would fit task=con_det, decoder=mag
Would fit task=con_det, decoder=peak


## 12. Individual-participant fitting branch

When `FIT_INDIVIDUAL = True`, the script loads participant-level OEI scores and fits model parameters for each participant.


In [20]:
# Schematic version of the participant-level update.
indiv_config = copy.deepcopy(config)

# For each participant:
# 1. select their behavioural OEI scores;
# 2. update indiv_config['OEI'][task];
# 3. keep only completed spatial-frequency conditions;
# 4. run the same optimisation pipeline as for average data.

## 13. Outputs saved by the script

For each model variant, task, decoder, and fitting level, the script saves several CSV files:

| File prefix | Contents |
|---|---|
| `optFit-...` | Final model output using the median parameter estimate from robust top fits. |
| `topFits-...` | Re-evaluated top candidate fits after filtering. |
| `summaryTopFits_...` | Summary statistics for the retained top fits. |

These outputs are designed to support later comparison across model variants and decoders.

## 14. Key things to check when running the script

Before running a large batch of fits, it is worth checking:

1. whether the `TITLE` is accurately named;
2. whether `FIT_AVERAGE`, `FIT_INDIVIDUAL`, and `FIT_BOTH_TASKS` are set as intended;
3. whether the behavioural CSV files exist in `beh_data`;
4. whether `n_level_reps`, `popsize`, and `maxiter` are appropriate for the desired speed and precision;
5. whether multiprocessing behaves as expected on the machine being used.

For debugging, reduce `TOTAL_N`, `popsize`, `maxiter`, and `n_level_reps` before running the full fitting batch.

## 15. Model overview

A compact way to understand the `ParamFits.py` script is:

```text
Choose model variant
    ↓
Choose task and decoder
    ↓
Simulate model using candidate parameters
    ↓
Compare simulated scores with behavioural targets
    ↓
Use differential evolution to reduce error
    ↓
Re-run the best candidates for more precise estimates of error
    ↓
Save final parameters and model outputs
```

The central idea is that parameter fitting is driven entirely by the objective error. Each candidate parameter set is used to simulate model behaviour. If the simulated scores move closer to the behavioural targets, the parameter set is treated as a better fit.

## 16. Plots
Please see `fitGraphs.py` to plot the results from `ParamFits.py`